# Lesson 3.6 — Groupby and Aggregation

**Objectives**
- Split data into groups with `.groupby()`
- Apply aggregation functions (`sum`, `mean`, `count`, custom functions)
- Produce summary tables from raw data

See `modules/03-pandas/notes.md` (Lesson 3.6) for the full written explanation.


In [1]:
import pandas as pd
from data_science_course.datasets import load_orders

orders = load_orders().drop_duplicates(subset=["order_id"])  # dedupe first, per Lesson 3.4
orders.shape

(6004, 11)

## A single aggregation

In [2]:
orders.groupby("status")["order_total"].mean().round(2)

status
cancelled    301.92
completed    272.78
returned     257.91
Name: order_total, dtype: float64

Split all rows by `status`, compute the mean of `order_total` within each group,
combine into one Series indexed by the group labels.

## Multiple aggregations at once

In [3]:
orders.groupby("status")["order_total"].agg(["count", "mean", "sum"]).round(2)

,count,mean,sum
status,,,
cancelled,192,301.92,57968.57
completed,5391,272.78,1470538.13
returned,421,257.91,108581.26


## Named aggregation -- clear output column names

In [4]:
summary = orders.groupby("status").agg(
    n_orders=("order_id", "count"),
    total_revenue=("order_total", "sum"),
    avg_order=("order_total", "mean"),
)
summary.round(2)

,n_orders,total_revenue,avg_order
status,,,
cancelled,192,57968.57,301.92
completed,5391,1470538.13,272.78
returned,421,108581.26,257.91


## Custom aggregation functions

In [5]:
def value_range(s):
    return s.max() - s.min()

orders.groupby("payment_method")["order_total"].agg(value_range).round(2)

payment_method
bank_transfer    4099.94
credit_card      3304.64
gift_card        3073.24
paypal           3454.64
Name: order_total, dtype: float64

Any function that takes a Series and returns one value works with `.agg()`,
built-in or custom.

## Grouping by multiple columns

In [6]:
orders.groupby(["status", "payment_method"])["order_total"].mean().round(2).head(6)

status     payment_method
cancelled  bank_transfer     349.49
           credit_card       266.40
           gift_card         310.11
           paypal            294.86
completed  bank_transfer     266.31
           credit_card       271.51
Name: order_total, dtype: float64

In [7]:
tidy = (
    orders.groupby(["status", "payment_method"])["order_total"]
    .mean()
    .round(2)
    .reset_index()
)
tidy.head()

,status,payment_method,order_total
0,cancelled,bank_transfer,349.49
1,cancelled,credit_card,266.40
2,cancelled,gift_card,310.11
3,cancelled,paypal,294.86
4,completed,bank_transfer,266.31


`.reset_index()` flattens the MultiIndex back into plain columns -- handy before
writing a summary table to disk or handing it to a plotting library.

## Try it yourself

1. Group `orders` by `payment_method` and compute the total number of orders and total
   revenue for each, using named aggregation.
2. Which `status` has the highest average `quantity` per order? (group by `status`,
   aggregate `quantity` with `mean`)
3. Write a custom aggregation function `pct_over_200` that returns the fraction of
   orders in a group with `order_total > 200`, then apply it grouped by `status`.
4. Group by both `status` and whether an order was discounted
   (`orders["discount_pct"] > 0`) and compute the mean `order_total` for each
   combination.


In [8]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [9]:
# 1.
by_payment = orders.groupby("payment_method").agg(
    n_orders=("order_id", "count"),
    total_revenue=("order_total", "sum"),
)
print(by_payment.round(2))

# 2.
avg_qty = orders.groupby("status")["quantity"].mean().round(2)
print(avg_qty)
print(avg_qty.idxmax())

# 3.
def pct_over_200(s):
    return (s > 200).mean()

print(orders.groupby("status")["order_total"].agg(pct_over_200).round(3))

# 4.
orders["is_discounted"] = orders["discount_pct"] > 0
print(orders.groupby(["status", "is_discounted"])["order_total"].mean().round(2))

                n_orders  total_revenue
payment_method                         
bank_transfer       1480      396018.18
credit_card         1494      409984.51
gift_card           1462      397507.20
paypal              1388      385284.08
status
cancelled    1.47
completed    1.52
returned     1.48
Name: quantity, dtype: float64
completed
status
cancelled    0.448
completed    0.394
returned     0.387
Name: order_total, dtype: float64
status     is_discounted
cancelled  False            308.99
           True             288.42
completed  False            282.18
           True             252.83
returned   False            254.47
           True             265.70
Name: order_total, dtype: float64
